🌐 [English](yolo26_od_pose_seg_depth.ipynb) | [한국어](yolo26_od_pose_seg_depth.ko.ipynb) | **日本語** | [中文](yolo26_od_pose_seg_depth.zh.ipynb)

# DEEPX Tutorial 21 - YOLO26 物体検出・姿勢推定・セグメンテーション・深度推定デモ

このチュートリアルでは、同じカメラまたは動画ストリームに対して、以下の 4 つの YOLO26s モデルを DEEPX NPU で実行する Qt ベースの C++ アプリケーションを解説します。
- YOLO26s 物体検出
- YOLO26s 姿勢推定
- YOLO26s インスタンスセグメンテーション
- YOLO26s 単眼深度推定

![YOLO マルチチャネルデモ](assets/yolo26-od-pos-seg-depth.png)

## 学習目標

このチュートリアルを終えると、次のことができるようになります。

- 自己完結型の C++ プロジェクト構造を理解する
- 検出、姿勢推定、セグメンテーション、深度推定の各パイプラインを識別する
- 1 つの入力フレームが 4 つの非同期ワーカーにどのように配布されるかを理解する
- Qt アプリケーションを Release モードでビルドする
- カメラまたは動画ファイルでデモを実行する

## このアプリケーションの NPU の使い方

| 項目 | このチュートリアルでは | 学んだ場所 |
|---|---|---|
| エンジン | タスクごとに 1 つ、合計 **4 つ**の `InferenceEngine` オブジェクトをすべて同じデバイス上で使用し、DX-RT がそれらのリクエストを NPU コア上でインターリーブする | T06-3 §2 |
| 実行 | 各ワーカーは `bufferCount` を自身のインフライト上限に設定して `RunAsync()` を呼び出し、完了コールバック内で描画する。最新フレームキューはレイテンシを積み上げる代わりに古いフレームを破棄する | T06-2 §5, §7 |
| タスクグラフ | 4 つのモデルはすべて `cpu_0` タスク(ONNX Runtime による CPU 上でのヘッドデコード)で終わるため、モデルを追加するたびにホスト CPU の負荷が増える | T06-1 §6 |
| リクエストあたりの入力 | 3 つのモデルは `[1, 640, 640, 3]` UINT8(1.2 MB)、深度モデルは `[1, 768, 768, 3]`(1.8 MB)を受け取る。1 枚のカメラフレームが 4 つのレターボックス済み入力になる | T06-3 §4 |
| 測定するもの | 3.1 節のモデルごとの `dxrun` ベースラインと、ウィンドウ内の 4 つの FPS ラベルを比較する。4 つのパネルのレートの合計が、NPU がどのように共有されているかを示す | T06-1 §5 |

## 前提条件

- チュートリアル 01 が完了していること: DX-RT がインストール済みで、DEEPX NPU が `/dev/dxrt0` として認識されていること。Qt ウィンドウ用に `cmake`、`g++`、`qtbase5-dev` が必要です。
- グラフィカルなデスクトップセッション。カメラデモ用の V4L2 カメラ(任意)。
- ダウンロード: 約 80 MB のアーカイブ 1 つ(深度モデルとサンプル動画、およびチュートリアル 01 がすでに共有ワークスペースに配置した 3 つの YOLO26s モデルのコピー)。
- 所要時間: 約 20 分。ビルドには 1〜2 分かかります。パッケージが不足している場合の第 2 節の `apt-get` 行を除き、`sudo` は不要です。
- DX-RT 3.4.2 で検証済みです。モデルは DX-COM 2.4.0 でコンパイルされています。

次のセルは SDK の場所を特定し、これらの要件を確認してステータス表を表示します。`MISSING` と表示された項目には、それを提供する手順が添えられます。

## 1. チュートリアルファイルの場所を確認する

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import shutil

paths = setup_tutorial(needs=["dx_rt", "npu", "cmake"])
paths.require(*["dx_rt", "npu", "cmake"])   # stop here with the fix if a prerequisite is missing
print()
print(f"Tutorial: {TUTORIAL_DIR}")
print(f"App:      {APP_DIR}")
print(f"Assets:   {ASSETS_DIR}")


### 1.1 プロジェクト構成

```text
T21-demo-yolo26-od-pose-seg-depth/
├── get_resources.sh
├── assets/
│   ├── models/
│   └── videos/
├── app/
│   ├── build.sh
│   ├── run_camera.sh
│   ├── run_video.sh
│   ├── CMakeLists.txt
│   ├── yolo26s_4.cpp
│   ├── common/
│   │   ├── base/
│   │   ├── processors/
│   │   └── utility/
│   ├── factory/
│   └── extern/
└── yolo26_od_pose_seg_depth.ipynb
```

In [ ]:
required_files = [
    TUTORIAL_DIR / "get_resources.sh",
    APP_DIR / "build.sh",
    APP_DIR / "run_camera.sh",
    APP_DIR / "run_video.sh",
    APP_DIR / "CMakeLists.txt",
    APP_DIR / "yolo26s_4.cpp",
    APP_DIR / "factory" / "yolo26s_factory.hpp",
    APP_DIR / "factory" / "yolo26s_pose_factory.hpp",
    APP_DIR / "factory" / "yolo26s_seg_factory.hpp",
]

for path in required_files:
    status = "OK" if path.is_file() else "MISSING"
    print(f"[{'OK' if path.is_file() else 'MISSING':7}] {path.relative_to(TUTORIAL_DIR)}")

## 2. 環境を確認する

Debian または Ubuntu のパッケージをインストールします。

```bash
sudo apt-get update
sudo apt-get install -y build-essential cmake pkg-config qtbase5-dev libopencv-dev ffmpeg v4l-utils
```

リソースアーカイブは約 84 MB です(モデル 4 つとサンプル動画 1 つ)。

In [ ]:
commands = ["g++", "cmake", "make", "qmake", "v4l2-ctl", "dxcli"]
for command in commands:
    location = shutil.which(command)
    print(f"[{'OK' if location else 'MISSING':7}] {command}")

device_nodes = sorted(Path("/dev").glob("dxrt*"))
print(f"\nDEEPX device nodes: {device_nodes if device_nodes else 'not found'}")

## 3. リソースをダウンロードして確認する

アプリケーションは `assets/` 配下に 4 つのモデルと 1 つのサンプル動画があることを前提としています。

```text
assets/
├── models/
│   ├── yolo26-s_640x640.dxnn
│   ├── yolo26-s-pose_640x640.dxnn
│   ├── yolo26-s-seg_640x640.dxnn
│   └── yolo26-depth-s_768x768_q-lite.dxnn
└── videos/
    └── dance-960-540.mp4
```

検出、姿勢推定、セグメンテーションの 3 つのモデルは、チュートリアル 01 の 3.3 節で共有ワークスペース(`<DX_ALL_SUITE_DIR>/workspace/res/models`)にダウンロードしたものと同じファイルなので、次のセルはそれらが存在する場合にそこからリンクします。深度モデルと動画はこのチュートリアルのアーカイブ(約 80 MB)にのみ含まれています。`get_resources.sh` がそれをダウンロードして `assets/` に展開し、展開が成功した後にアーカイブを削除します。

In [ ]:
SHARED_MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"      # filled by Tutorial 01 section 3.3
MODEL_NAMES = [
    "yolo26-s_640x640.dxnn",
    "yolo26-s-pose_640x640.dxnn",
    "yolo26-s-seg_640x640.dxnn",
    "yolo26-depth-s_768x768_q-lite.dxnn",
]
default_assets = [ASSETS_DIR / "models" / name for name in MODEL_NAMES]
(ASSETS_DIR / "models").mkdir(parents=True, exist_ok=True)

# Reuse the shared-workspace copies instead of keeping a second 60 MB copy here.
for path in default_assets[:3]:
    shared = SHARED_MODEL_DIR / path.name
    if path.is_symlink() and not path.exists():
        path.unlink()   # stale link
    if not path.exists() and shared.is_file():
        path.symlink_to(shared)
        print(f"Linked from the shared workspace: {path.name}")

for path in default_assets:
    print(f"[{'OK' if path.is_file() else 'MISSING':7}] {path.relative_to(TUTORIAL_DIR)}")

video_files = sorted((ASSETS_DIR / "videos").glob("*"))
video_files = [path for path in video_files if path.is_file() and path.name != '.gitkeep']
print(f"\nAvailable video files: {len(video_files)}")
for path in video_files[:10]:
    print(f"  - {path.name}")

resources_missing = any(not path.is_file() for path in default_assets) or not video_files


次のセルはリソースが不足している場合にのみ実行してください。同じ名前の既存ファイルは展開時に置き換えられることがあります。

In [ ]:
if resources_missing:
    !cd "{TUTORIAL_DIR}" && ./get_resources.sh
else:
    print("All required resources are already available.")

video_files = sorted((ASSETS_DIR / "videos").glob("*"))
video_files = [path for path in video_files if path.is_file() and path.name != '.gitkeep']
for path in default_assets:
    print(f"[{'OK' if path.is_file() else 'MISSING':7}] {path.relative_to(TUTORIAL_DIR)}")
print(f"Available video files: {len(video_files)}")

missing_after_setup = [path for path in default_assets if not path.is_file()]
if missing_after_setup:
    missing_text = "\n".join(f"  - {path}" for path in missing_after_setup)
    raise FileNotFoundError(
        "Required model files are still missing after resource setup:\n" + missing_text
    )
if not video_files:
    raise FileNotFoundError("No input video was found under assets/videos.")

### 3.1 4 つのモデルを検査してベンチマークする

アプリケーションを読む前に、4 つのエンジンが実行するものを確認しましょう。`dxparse -v` を見ると、すべてのモデルが NPU タスクの後に `cpu_0` タスクを持つことがわかります。検出ヘッド(および深度モデルの出力ステージ)は DX-RT 内部の ONNX Runtime でデコードされるため、意味のあるベンチマークには `--use-ort` が必要です。`dxrun` の数値は単一モデルのベースラインです。アプリケーションは同じ NPU 上で 4 つすべてを同時に実行するため、各パネルの FPS はそのベースラインより低くなります。

```bash
cd <T21>/assets/models
dxparse -m yolo26-s_640x640.dxnn -v
dxrun -m yolo26-s_640x640.dxnn --use-ort -t 3 -v
```

In [ ]:
for path in default_assets:
    print(f"\n===== {path.name} =====")
    !dxparse -m "{path}" -v 2>&1 | grep -E "Task\[|^\s+-\s+\S+, (UINT8|FLOAT)"
    !dxrun -m "{path}" --use-ort -t 3 -v 2>&1 | grep -E "Task\[|FPS|Latency|NPU Processing" 


## 4. アプリケーションアーキテクチャ

```text
Camera or video
       |
       v
CaptureThread
       |
       +--> Detection worker    --> Object Detection panel
       +--> Pose worker         --> Pose Estimation panel
       +--> Segmentation worker --> Instance Segmentation panel
       +--> Depth worker        --> Depth Estimation panel
```

キャプチャスレッドは各 BGR フレームを 4 つの最新フレームキューに公開します。検出、姿勢推定、セグメンテーションはタスク固有のファクトリを使用します。`DepthWorker` は 768 x 768 のレターボックス前処理、非同期 DXRT 推論、レターボックスの除去、ソースフレームへのリサイズ、Turbo カラーマッピングを行います。Qt は 4 つのライブ結果をすべて全画面の 2 x 2 グリッドに表示します。

## 5. C++ コードを読む

次のヘルパーは、現在のソースファイルから選択したセクションを表示します。

In [ ]:
from IPython.display import Code, display

def show_source(relative_path, marker, line_count=80):
    path = APP_DIR / relative_path
    lines = path.read_text(encoding="utf-8").splitlines()
    try:
        start = next(index for index, line in enumerate(lines) if marker in line)
    except StopIteration as error:
        raise ValueError(f"Marker not found in {relative_path}: {marker}") from error

    end = min(start + line_count, len(lines))
    print(f"{relative_path}:{start + 1}-{end}")
    display(Code("\n".join(lines[start:end]), language="cpp"))

### 5.1 ビルド構成

CMake は 1 つの C++17 実行ファイルをビルドし、Qt5 Widgets、OpenCV、DXRT をリンクします。`PROJECT_ROOT_DIR` はチュートリアルディレクトリを指し、既定のモデルパスが `assets/models/` 配下に解決されるようにします。

In [ ]:
show_source("CMakeLists.txt", "find_package(OpenCV", line_count=55)

### 5.2 コマンドラインオプションと既定のリソース

`AppArgs` は 4 つのモデルパス、カメラ設定、任意の動画パス、デバッグオプションを定義します。`--model-depth` は既定の深度モデルを上書きします。`--video` を省略するとカメラモードになります。V4L2 デバイスを選ぶには `-c` または `--camera` を、キャプチャ設定を要求するには `--width`、`--height`、`--fps` を使用します。これらのオプションを省略した場合の既定値は `/dev/video0`、1280 x 720、30 FPS です。

In [ ]:
show_source("yolo26s_4.cpp", "struct AppArgs", line_count=48)
show_source("yolo26s_4.cpp", "AppArgs parseArgs", line_count=70)

### 5.3 タスクファクトリ

各ファクトリは、1 つのタスクに対して適切な前処理、後処理、可視化コンポーネントを生成します。

In [ ]:
show_source("factory/yolo26s_factory.hpp", "class Yolo26sFactory", line_count=42)
show_source("factory/yolo26s_pose_factory.hpp", "class Yolo26s_poseFactory", line_count=42)
show_source("factory/yolo26s_seg_factory.hpp", "class Yolo26s_segFactory", line_count=42)

### 5.4 非同期結果ワーカー

各ワーカーは独自の `InferenceEngine` を生成し、非同期コールバックを登録します。検出、姿勢推定、セグメンテーションはタスク固有のファクトリを使用します。専用の深度ワーカーは `[1, 768, 768, 3]` UINT8 入力と `[1, 1, 768, 768]` FLOAT 出力のコントラクトを検証し、コールバックが完了するまで入力バッファを保持し、レターボックスのパディングを除去し、ソースフレームのジオメトリを復元して、OpenCV の Turbo カラーマップを適用します。各最新フレームキューは、モデルが入力ストリームより遅い場合にレイテンシを積み上げる代わりに古いフレームを置き換えます。

In [ ]:
show_source("yolo26s_4.cpp", "void ResultWorker<ResultT, FactoryT>::run()", line_count=92)
show_source("yolo26s_4.cpp", "class DepthWorker final", line_count=82)
show_source("yolo26s_4.cpp", "void DepthWorker::preprocess", line_count=115)
show_source("yolo26s_4.cpp", "void DepthWorker::run()", line_count=125)

### 5.5 カメラと動画のキャプチャ

`CaptureThread` は OpenCV の `VideoCapture` を使用します。動画入力は `--no-loop-video` が設定されていない限りループ再生されます。カメラ入力は V4L2 バックエンドを使用し、MJPG 形式を要求します。

In [ ]:
show_source("yolo26s_4.cpp", "void CaptureThread::runVideo()", line_count=65)
show_source("yolo26s_4.cpp", "void CaptureThread::runCamera()", line_count=65)

### 5.6 Qt 2 x 2 ウィンドウ

`QuadWindow` は 4 つのライブパネルを生成し、4 つの推論ワーカーとキャプチャスレッドを起動し、FPS ラベルを更新し、整然とシャットダウンを行います。

In [ ]:
show_source("yolo26s_4.cpp", "class QuadWindow final", line_count=88)

## 6. アプリケーションをビルドする

`build.sh` は `app/build/` を作成し、CMake を Release モードで構成し、`nproc` が報告するすべての CPU コアを使って `make` を実行します。完全な再ビルドには `./build.sh --clean` を、DX-RT がカスタムプレフィックス配下にインストールされている場合は `./build.sh -DCMAKE_PREFIX_PATH=<prefix>` を使用してください。

In [ ]:
!cd "{APP_DIR}" && ./build.sh

binary_path = APP_DIR / "build" / "yolo26s_4"
print(f"Executable exists: {binary_path.is_file()}")


## 7. カメラデモを実行する

既定のデバイスは `/dev/video0` で、1280 x 720 と 30 FPS を要求します。別のカメラやキャプチャ設定を選ぶには、下の変数を変更してください。実行セルは全画面の Qt ウィンドウを開き、アプリケーションが終了するまでブロックします。

In [ ]:
CAMERA_DEVICE = "/dev/video0"
CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720
CAMERA_FPS = 30

camera_path = Path(CAMERA_DEVICE)
print(f"Camera exists: {camera_path.exists()}")
if shutil.which("v4l2-ctl"):
    !v4l2-ctl --list-devices

In [ ]:
RUN_CAMERA = False   # set to True to open the GUI; the cell stays busy until the window is closed

if not binary_path.is_file():
    raise FileNotFoundError("Build the application before running the demo.")
if not camera_path.exists():
    raise FileNotFoundError(f"The camera is not available: {CAMERA_DEVICE}")
if RUN_CAMERA:
    !cd "{APP_DIR}" && ./run_camera.sh --camera "{CAMERA_DEVICE}" --width {CAMERA_WIDTH} --height {CAMERA_HEIGHT} --fps {CAMERA_FPS}
else:
    print("Set RUN_CAMERA = True to start the camera GUI.")


## 8. 動画デモを実行する

`VIDEO_PATH` に `assets/videos/` 配下のファイルを設定します。`run_video.sh` は最初の引数として動画パスを受け取ります。

In [ ]:
VIDEO_PATH = video_files[0] if video_files else None
print(f"Selected video: {VIDEO_PATH if VIDEO_PATH else 'no video available'}")

In [ ]:
RUN_VIDEO = False   # set to True to open the GUI; the cell stays busy until the window is closed

if not binary_path.is_file():
    raise FileNotFoundError("Build the application before running the demo.")
if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Set VIDEO_PATH to an existing video file.")
if RUN_VIDEO:
    !cd "{APP_DIR}" && ./run_video.sh "{VIDEO_PATH}"
else:
    print("Set RUN_VIDEO = True to start the video GUI.")


## 9. 操作方法

- `Esc` または `q`: 終了
- `EXIT` ボタン: マウスで終了

## 10. トラブルシューティング

| 症状 | 表示される内容 | 考えられる原因 | 対処 |
|---|---|---|---|
| CMake が Qt5 を見つけられない | `Could not find a package configuration file provided by "Qt5"` | `qtbase5-dev` がインストールされていない | `sudo apt install qtbase5-dev` |
| CMake が DXRT を見つけられない | `Could not find a package configuration file provided by "dxrt"` | DX-RT がインストールされていないか、カスタムプレフィックス配下にある | DX-Runtime をインストールする(チュートリアル 01 第 3 節)か、`./build.sh -DCMAKE_PREFIX_PATH=<prefix>` を実行する |
| モデルがロード時に拒否される | DX-RT のバージョンまたは形式のエラー | DXNN が別の SDK リリース向けにコンパイルされた(これらのモデルは DX-COM 2.4.0) | このチュートリアルが検証された DX-RT バージョン(3.4.2)を使用するか、お使いの DX-COM で再コンパイルする |
| モデルを開けない | `assets/models` 配下のファイル名を示すロードエラー | リソースが不足している | 第 3 節のリソースセルを実行する |
| カメラを開けない | `cannot open /dev/video0` | カメラがないか、権限がない | `v4l2-ctl --list-devices` を実行し、ユーザーを `video` グループに追加し、`--camera` で別のデバイスを指定する |
| Qt ウィンドウが表示されない | `could not connect to display` | グラフィカルセッションがない | デスクトップセッションで実行する |

## 11. まとめ

1 つのキャプチャスレッドが各フレームを 4 つの独立した非同期 DXRT パイプラインに送ります。タスク固有のファクトリが物体検出、姿勢推定、インスタンスセグメンテーションを処理します。専用の深度ワーカーはフレームをレターボックス処理し、768 x 768 の深度モデルを実行し、深度マップをソースのジオメトリに復元して、Turbo カラーマップを適用します。Qt は 4 つのライブ結果をすべて全画面の 2 x 2 レイアウトに表示します。

### 11.1 完了チェックリスト

- [ ] 4 つのモデルとサンプル動画をダウンロードする
- [ ] 1 つのキャプチャスレッドが 4 つの非同期パイプラインに入力を供給する仕組みを読む
- [ ] Qt アプリケーションをビルドする
- [ ] 動画デモとカメラデモを実行する

> **次へ:** チュートリアル 22 に進み、CLIP でカメラフレームをテキストクエリと照合してみましょう。